In [ ]:
import pandas as pd

df = pd.read_csv('/content/sql_create_HW - Sheet1.csv')

In [ ]:
df.info()
print(f'\nКількість рядків: {len(df)}')
print(f'Список колонок: {df.columns.tolist()}')
print('\nКількість пропусків в колонках:')
print(df.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   full_name         8 non-null      object 
 1   email             8 non-null      object 
 2   signup_date       8 non-null      object 
 3   last_login        7 non-null      object 
 4   is_active         7 non-null      object 
 5   gpa               8 non-null      float64
 6   enrolled_classes  8 non-null      object 
 7   scholarships      8 non-null      object 
dtypes: float64(1), object(7)
memory usage: 644.0+ bytes

Кількість рядків: 8
Список колонок: ['full_name', 'email', 'signup_date', 'last_login', 'is_active', 'gpa', 'enrolled_classes', 'scholarships']

Кількість пропусків в колонках:
full_name           0
email               0
signup_date         0
last_login          1
is_active           1
gpa                 0
enrolled_classes    0
scholarships        0
dtype: int64


In [ ]:
df['signup_date'] = pd.to_datetime(df['signup_date']).dt.date
df['last_login'] = pd.to_datetime(df['last_login'], errors='coerce')
i_bool = {
    'TRUE': True, 'True': True, '1': True, 1: True, 'Yes': True, 'yes': True,
    'FALSE': False, 'False': False, '0': False, 0: False, 'No': False, 'no': False
}
df['is_active'] = df['is_active'].map(i_bool).fillna(False).astype(bool)
df['gpa'] = pd.to_numeric(df['gpa'], errors='coerce').astype(float)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   full_name         8 non-null      object        
 1   email             8 non-null      object        
 2   signup_date       8 non-null      object        
 3   last_login        7 non-null      datetime64[ns]
 4   is_active         8 non-null      bool          
 5   gpa               8 non-null      float64       
 6   enrolled_classes  8 non-null      object        
 7   scholarships      8 non-null      object        
dtypes: bool(1), datetime64[ns](1), float64(1), object(5)
memory usage: 588.0+ bytes


/tmp/ipython-input-19691743.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['is_active'] = df['is_active'].map(i_bool).fillna(False).astype(bool)


In [ ]:
df = df.sort_values(by=['is_active', 'last_login', 'gpa'], ascending=False)
df = df.drop_duplicates(subset=['email'], keep='first')

In [ ]:
invalid_signup = (df['last_login'] < df['signup_date']) | (df['last_login'].isna())

report = df[invalid_signup][['email', 'last_login', 'signup_date']].copy()
report = report.rename(columns={'last_login': 'old_last_login'})
report['new_last_login'] = pd.NaT

new_login = report['old_last_login'].notna()
report.loc[new_login, 'new_last_login'] = pd.to_datetime(report.loc[new_login, 'signup_date'])

report['reason'] = 'Невалідна дата'
report.loc[report['old_last_login'].isna(), 'reason'] = 'Користувач не логінився'

print(report.to_string(index=False))

#df.loc[invalid_signup, 'last_login'] = df.loc[invalid_signup, 'signup_date']

               email      old_last_login signup_date new_last_login                  reason
george.n@example.com 2025-06-29 15:00:00  2025-07-01     2025-07-01          Невалідна дата
 diana.p@example.com                 NaT  2024-11-01            NaT Користувач не логінився


In [ ]:
median = df['gpa'].median()
report = df[(df['gpa'] < 0) | (df['gpa'] > 4) | (df['gpa'].isna())][['email', 'gpa']].copy()
df['gpa'] = df['gpa'].clip(0, 4).fillna(median)
report['new_gpa'] = df.loc[report.index, 'gpa']
print(report.to_string(index=False))

              email  gpa  new_gpa
diana.p@example.com  4.2      4.0


In [ ]:
df.to_csv('sql_create_HW_cleaned.csv', index=False)

In [ ]:
sql_values = df.apply(lambda r:
    f"('{r['full_name']}', '{r['email']}', '{r['signup_date']}', "
    f"{f"'{r['last_login']}'" if pd.notna(r['last_login']) else 'NULL'}, "
    f"{str(r['is_active']).upper()}, {r['gpa']}, '{r['enrolled_classes']}', '{r['scholarships']}')",
    axis=1
)
print(",\n".join(sql_values) + ";")

('George Novak', 'george.n@example.com', '2025-07-01', '2025-06-29 15:00:00', TRUE, 2.65, '{BIO101}', '{Athletic Scholarship}'),
('Alice Johnson', 'alice.j@example.com', '2024-09-01', '2025-06-28 14:30:00', TRUE, 3.85, '{CS101,MATH202,ENG150}', '{Merit Scholarship}'),
('Fatima Ali', 'fatima.a@example.com', '2025-02-10', '2025-06-27 11:22:00', TRUE, 3.75, '{ECON101,BUS200}', '{Leadership Award,Women in Business}'),
('Evan Lee', 'evan.l@example.com', '2025-01-20', '2025-06-25 08:00:00', TRUE, 3.0, '{MATH101,CS101}', '{}'),
('Carlos Nguyen', 'carlos.n@example.com', '2024-10-15', '2025-06-20 19:45:00', TRUE, 3.45, '{CS101,STAT200,PHYS105}', '{STEM Grant}'),
('Bob Smith', 'bob.smith@example.com', '2024-09-03', '2025-06-30 09:10:00', FALSE, 2.95, '{HIST110,PHIL101}', '{}'),
('Diana Petros', 'diana.p@example.com', '2024-11-01', NULL, FALSE, 4.0, '{ART101,DES202}', '{Dean's List}');
